## **Install & imports**. Standard setup; requests is here to fetch the corpus

In [ ]:
!pip install -q transformers datasets

import torch
import requests
from transformers import AutoModelForCausalLM, AutoTokenizer

## **Load the base model** and tokenizer, put the model on the GPU.

In [ ]:
model_id = "HuggingFaceTB/SmolLM-135M"     # BASE, not -Instruct

device    = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model     = AutoModelForCausalLM.from_pretrained(model_id).to(device)

print(f"Loaded {model_id} on {device} — {sum(p.numel() for p in model.parameters()):,} params")

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.69k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  538MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Loaded HuggingFaceTB/SmolLM-135M on cuda — 134,515,008 params


## **Download the domain corpus**. Raw text is all CPT needs — no labels, no pairs, just a body of text in the target domain.

In [ ]:
url  = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text
print(f"Corpus length: {len(text):,} chars")
print(text[:200])

Corpus length: 1,115,394 chars
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


## **Tokenize the whole corpus into one long stream of token IDs** (this is "packing"). We encode the entire text once, end to end, into a flat tensor of IDs.

In [ ]:
ids  = tokenizer(text, add_special_tokens=False)["input_ids"]   # list[int]
data = torch.tensor(ids, dtype=torch.long)
print(f"Total tokens: {len(data):,}")

Total tokens: 341,094


## **Train/val split + the batch loader.**

In [ ]:
n          = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

block_size = 256
batch_size = 8

def get_batch(split):
    d  = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    xb = torch.stack([d[i : i + block_size] for i in ix])
    return xb.to(device)          # note: we return ONLY x, no shifted y

## **Hyperparameters**

In [ ]:
lr           = 5e-5      # much smaller than pretraining — we NUDGE, not overwrite
max_steps    = 300
eval_every   = 50
weight_decay = 0.01
grad_clip    = 1.0

## **Optimizer + the training loop**

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

@torch.no_grad()
def eval_loss(batches=10):
    model.eval()
    total = 0.0
    for _ in range(batches):
        xb    = get_batch('val')
        total += model(input_ids=xb, labels=xb).loss.item()
    model.train()
    return total / batches

model.train()
for step in range(max_steps):
    xb   = get_batch('train')
    loss = model(input_ids=xb, labels=xb).loss          # HF computes next-token CE

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)   # NEW
    optimizer.step()

    if step % eval_every == 0 or step == max_steps - 1:
        print(f"step {step:4d} | train {loss.item():.4f} | val {eval_loss():.4f}")

step    0 | train 3.7773 | val 3.5711
step   50 | train 3.7171 | val 3.4670
step  100 | train 3.6939 | val 3.4628
step  150 | train 3.4796 | val 3.4722
step  200 | train 3.3792 | val 3.3922
step  250 | train 3.4196 | val 3.3859
step  299 | train 3.4389 | val 3.4435


## **A generate helper** (run it before AND after training)

In [ ]:
def sample(prompt, max_new_tokens=120):
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=True, temperature=0.7, top_p=0.9,
        repetition_penalty=1.3,        # your v1/v2 rep_penalty, now a generate() arg
        no_repeat_ngram_size=3,        # hard-bans repeating any exact 3-token span
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(sample("Enter the king, and he speaks to his son:"))

Enter the king, and he speaks to his son:
"Thou art a good prince; thou hast chosen a wise king. Now come again, to the place where thy father dwelt."
And he said unto him this after:
I am not at home with thee now; go in thyself into the country of the Lions for that I may know your state well enough before you are released from it hereafter--and therefore give me thy son's name as thy wife's daughter, lest thou shouldest be called thy daughter's brother: but do so also by my own consent or mine uncle's. For I will have


### Before CPT Output:
To be, or not to be, is a question of choice. If you want to live in the city, but not to live in the city, then it is a choice. If you want to live in the city, but not to live in the city, then it is not a choice. If you want to live in the city, but not to live in the city, then it is not a choice.

So, the question of whether or not to live in the city is not a question of choice, but a question of choice. If you want to live in the city, but not to live in

---

### After CPT Output:
Enter the king, and he speaks to his son:

"Thou art a good prince; thou hast chosen a wise king. Now come again, to the place where thy father dwelt."

And he said unto him this after:

I am not at home with thee now; go in thyself into the country of the Lions for that I may know your state well enough before you are released from it hereafter--and therefore give me thy son's name as thy wife's daughter, lest thou shouldest be called thy daughter's brother: but do so also by my own consent or mine uncle's. For I will have



## **Save the fine-tuned model**. Persist it in HF format so you can reload with `from_pretrained`.

In [ ]:
model.save_pretrained("smollm-cpt-shakespeare")
tokenizer.save_pretrained("smollm-cpt-shakespeare")
print("Saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved.




---


## **SFT** From the Scratch:

---



In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

### **Build pairs from the Shakespeare corpus:**

In [ ]:
TEMPLATE = ("### Instruction:\nContinue the passage in the style of Shakespeare.\n\n"
            "### Passage:\n{prompt}\n\n### Response:\n")

def make_pairs(text, n_pairs=2000, prompt_chars=200, resp_chars=200):
    pairs, step = [], (len(text) - prompt_chars - resp_chars) // n_pairs
    for k in range(n_pairs):
        i = k * step
        prompt = TEMPLATE.format(prompt=text[i : i+prompt_chars])
        resp   = text[i+prompt_chars : i+prompt_chars+resp_chars]
        pairs.append((prompt, resp))
    return pairs

pairs = make_pairs(text)

### **Tokenize one pair into input_ids + masked labels:**

In [ ]:
EOS = tokenizer.eos_token_id

def build_example(prompt_text, response_text):
    prompt_ids   = tokenizer(prompt_text,   add_special_tokens=False)["input_ids"]
    response_ids = tokenizer(response_text, add_special_tokens=False)["input_ids"] + [EOS]
    input_ids = prompt_ids + response_ids
    labels    = [-100] * len(prompt_ids) + response_ids   # ← mask the prompt, keep the response (+EOS)
    return input_ids, labels

### **Collate a batch —** pad and mask the padding too:

In [ ]:
PAD = tokenizer.pad_token_id

def get_sft_batch(bs=4):
    import random
    ex = [build_example(p, r) for p, r in random.sample(pairs, bs)]
    maxlen = max(len(ids) for ids, _ in ex)
    input_ids, labels, attn = [], [], []
    for ids, lab in ex:
        pad = maxlen - len(ids)
        input_ids.append(ids + [PAD]  * pad)
        labels.append(   lab + [-100] * pad)     # padding never contributes to loss
        attn.append(     [1]*len(ids) + [0]*pad) # padding mask: 1 real, 0 pad
    t = lambda z: torch.tensor(z).to(device)
    return t(input_ids), t(labels), t(attn)

### **Training loop:**

In [ ]:
for step in range(max_steps):
    input_ids, labels, attn = get_sft_batch(bs=4)
    loss = model(input_ids=input_ids, attention_mask=attn, labels=labels).loss  # ← labels ≠ input_ids now; attn added

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()
    if step % eval_every == 0:
        print(f"step {step:4d} | loss {loss.item():.4f}")

step    0 | loss 4.5549
step   50 | loss 3.8835
step  100 | loss 3.4139
step  150 | loss 3.6482
step  200 | loss 3.7995
step  250 | loss 4.3892


### **Test it**

In [ ]:
def sft_generate(passage):
    prompt = TEMPLATE.format(prompt=passage)
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(**enc, max_new_tokens=120, do_sample=True,
                         temperature=0.7, top_p=0.9, repetition_penalty=1.3,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

print(sft_generate("The night is dark and full of"))

black darkness, a great sea; he has no sun to shine on him! The moon does not rise till morning nor sunset until noon or midnight."


### Output after SFT:
black darkness, a great sea; he has no sun to shine on him! The moon does not rise till morning nor sunset until noon or midnight."
